In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# import from the Python CRUD Module using the AnimalShelter method
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Username and password
username = "aacuser"
password = "1234"
shelter = AnimalShelter(username, password)

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Added HTLM tag and unique identifier
app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'width': '250px', 'padding-bottom': '10px'}),
        
        # Unique identifier
        html.H3('Dashboard created by Connor Casey')
    ]),
    html.Hr(),
    html.Div(
        # Radio buttons for interactive filtering
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset', 'value': 'Reset'}
            ],
            value='Reset', # Default value upon load
            labelStyle={'display': 'inline-block', 'padding': '10px'}
        )
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # Features for interactive data table
        editable=False,
        sort_action="native",    # Allows sorting by column
        sort_mode="multi",       # Allows sorting multiple columns
        column_selectable=False,
        row_selectable="single",    
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='map-id',
            className='col s12 m6',
            ),
                 
        html.Div(
            id='graph-id',
            className='col s12 m6',

            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## Default Query (Reset)
    query = {}
    
    if filter_type == 'Water':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Mountain':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
    elif filter_type == 'Disaster':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
    
    # Query the db
    df = pd.DataFrame.from_records(db.read(query))
    
    # If query returns no results, create empty dataframe with correct columns
    if df.empty:
        df = pd.DataFrame(columns = ['animal_id', 'animal_type', 'breed', 'color', 'name', 'age_upon_outcome_in_weeks', 'sex_upon_outcome', 'location_lat', 'location_long'])
    else:
        # Drop the ObjectID column
        if '_id' in df.columns:
            df.drop(columns=['_id'], inplace=True)
    
    # Return the dictionary for the datatable
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # If data is empty
    if viewData is None or len(viewData) == 0:
        return html.Div("Loading chart...")
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # If table is empty
    if dff.empty:
        return html.Div("No data available for chart.")
    
    # Create a copy of the df for plotting
    plot_df = dff.copy()
    
    # Grouping for the Pie Chart so there are not too many slices when no filter/"Reset" option is selected
    # If there are more than 10 unique breeds, apply the grouping.
    # If there are 10 or fewer, do not alter
    if plot_df['breed'].nunique() > 10:
        top_10 = plot_df['breed'].value_counts().nlargest(10).index
        plot_df.loc[~plot_df['breed'].isin(top_10), 'breed'] = 'Other'
    
    # Generate the pie chart
    fig = px.pie(plot_df, names='breed', title='Preferred Breeds')
    
    return [
        dcc.Graph(figure=fig)
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Check if the table is empty
    if dff.empty:
        return html.Div("No animals match the current filter.")
    
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
                        
    # Check if filtering causes the selected row index to be out of bounds
    if row >= len(dff):
        row = 0
        
    # Get lat and lon from the selected row
    lat = dff.iloc[row, 13]
    lon = dff.iloc[row, 14]
    
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[lat, lon], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://gloriavega-hydroavenue-3000.codio.io/proxy/8050/
